<div style="
background-color:#EAEAEA;
padding:20px;
border-left:5px solid #6C757D;
border-radius:6px;">

<table style="width:100%; border:none;">
<tr style="border:none;">

<td style="border:none; vertical-align:top;">

<h1 style="font-size:32px; margin-top:0;">
Master's Thesis
</h1>

<hr style="margin:16px 0 22px 0;">

<p style="font-size:22px; line-height:1.5; margin:0;">
<strong>Master's Degree in Advanced Physics</strong> - 
<strong>Universitat de Val?ncia</strong>
</p>

<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">
This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:
</p>

<div style="
font-size:25px;
font-weight:700;
line-height:1.3;
margin-top:14px;
margin-bottom:26px;">
Fast Simulation of Neutrino Oscillations in Matter
</div>

<p style="font-size:14px; line-height:1.55;">
<strong>Author</strong><br>
Juan Ramon Diaz Santos - 
<a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a>
</p>

<p style="font-size:14px; line-height:1.55;">
<strong>Supervisors</strong><br>
Roberto Ruiz de Austri Bazan ?
<a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>
Michele Lucente ?
<a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a>
</p>

<p style="font-size:14px; line-height:1.55; margin-bottom:0;">
<strong>Date</strong><br>
September 2026
</p>

</td>

<td style="
border:none;
width:230px;
padding-left:25px;
text-align:right;
vertical-align:top;">

<img src="../logo_uv.png"
     alt="Universitat de Val?ncia"
     style="width:200px; margin-top:5px;">

</td>

</tr>
</table>

</div>

# Benchmark 1 — Perturbative versus Numerical Propagation (Google Colab)
---

This Google Colab variant compares execution time and speedup of the perturbative (`analytical`) and segmented matrix-exponential (`numerical`) methods over the same four physical pipelines used by the intrinsic-validation notebook. It installs Tpeanuts in Colab's local filesystem and stores generated outputs persistently in Google Drive.

---

## Table of Contents

| # | Section |
|---|---------|
| [0](#0.-Theory-Background) | **Theory Background**: benchmark metric, synchronization and statistical estimator |
| [1](#1.-Colab-Setup-and-Libraries) | **Colab Setup and Libraries** |
| [2](#2.-Paths-and-Configuration) | **Paths and Configuration** |
| [3](#3.-Pure-Flavour-States-through-Earth) | **Pure Flavour States through Earth** |
| [4](#4.-Solar-Neutrinos-to-the-Detector) | **Solar Neutrinos to the Detector** |
| [5](#5.-Atmospheric-Neutrinos-to-the-Surface) | **Atmospheric Neutrinos to the Surface** |
| [6](#6.-Atmospheric-Neutrinos-to-the-Detector) | **Atmospheric Neutrinos to the Detector** |
| [7](#7.-Resource-Consumption) | **Resource Consumption** |
| [∑](#8.-Summary) | **Summary** |

## 0. Theory Background

### 0.1 Compared algorithms

The perturbative method evaluates exact constant-density evolution and a first-order residual-density correction inside each fitted material segment. The numerical method samples the trajectory and composes $S_j=\exp(-iH_j\Delta x_j)$ over many short intervals. Both implement the same Schrödinger equation in matter *(Wolfenstein 1978; Mikheyev & Smirnov 1985)* but have different computational costs.

### 0.2 Timing and speedup

For each pair of energy- and angular-grid sizes, elapsed wall time is measured after one warm-up execution. GPU work is asynchronous, so CUDA is synchronized immediately before and after every timed call. The reported estimator is the median of repeated executions,

$$t_m=\mathrm{median}(t_{m,1},\ldots,t_{m,N}),\qquad \mathcal{S}=\frac{t_{\rm numerical}}{t_{\rm perturbative}}.$$

Thus $\mathcal{S}>1$ means that perturbative propagation is faster. Timing includes construction of the medium trajectory/profile and probability projection, but excludes plotting and comparison-table construction.

### 0.3 Fair-comparison conditions

Both branches receive identical energy and angular grids, chunk boundaries, oscillation parameters, density sources and detector geometry. The default Colab scan uses $N_E,N_\theta\in\{4,8,16,32\}$ to keep execution practical on shared runtimes. Earth reunitarization is enabled for the perturbative branch. The numerical discretizations use 400 Earth steps and 600 atmosphere steps. Chunking changes peak memory and dispatch overhead but not the mathematical grid evaluated by either method.

### References

- L. Wolfenstein, *Neutrino oscillations in matter*, Phys. Rev. D **17**, 2369 (1978).
- S. P. Mikheyev and A. Yu. Smirnov, *Resonant amplification of neutrino oscillations in matter*, Sov. J. Nucl. Phys. **42**, 913 (1985).
- CUDA Programming Guide, *Asynchronous concurrent execution* — device synchronization requirements for host-side timing.

## 1. Colab Setup and Libraries

### 1.1 Colab environment

This bootstrap cell mounts Google Drive, clones Tpeanuts into Colab's fast local storage, installs the project with the active kernel interpreter, and redirects all notebook outputs to Drive. Set `USE_GOOGLE_DRIVE=False` to use ephemeral `/content/output` storage instead.

**Expected results:** the repository is available at `/content/tpeanuts`, dependencies are installed in the active Colab kernel, and the selected output root is printed.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if not IN_COLAB:
    raise RuntimeError('This notebook variant is intended for Google Colab.')

USE_GOOGLE_DRIVE = True
REPOSITORY_URL = 'https://github.com/juanramondiaz/tpeanuts.git'
REPOSITORY_DIR = Path('/content/tpeanuts')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/TPeanuts/output')
else:
    OUTPUT_ROOT = Path('/content/output')

if not REPOSITORY_DIR.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
os.chdir(REPOSITORY_DIR)
if str(REPOSITORY_DIR.parent) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIR.parent))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(REPOSITORY_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'psutil', 'nvidia-ml-py'], check=True)
os.environ['TPEANUTS_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Python      : {sys.executable}')
print(f'Repository  : {REPOSITORY_DIR}')
print(f'Output root : {OUTPUT_ROOT}')

### 1.2 Libraries

In [ ]:
from __future__ import annotations

%matplotlib inline
import os

# These limits must be defined before importing NumPy or PyTorch. Restart the
# kernel after changing them so that every native backend observes them.
CPU_THREADS = min(4, os.cpu_count() or 1)
TORCH_INTEROP_THREADS = 1
CUDA_DEVICE = '0'  # Set to '' to force CPU execution.
for variable in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ[variable] = str(CPU_THREADS)
os.environ['CUDA_VISIBLE_DEVICES'] = CUDA_DEVICE

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import psutil
import torch

torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(TORCH_INTEROP_THREADS)
except RuntimeError:
    # PyTorch only permits changing this setting before inter-op work starts.
    if torch.get_num_interop_threads() != TORCH_INTEROP_THREADS: raise

from tpeanuts.notebooks.notebookConfig import load_notebook_config
from tpeanuts.notebooks.notebooks_helper import save_and_show
from tpeanuts.notebooks.benchmark.benchmark_helpers import iter_grid_chunks, plot_grid_resource_scaling, reduce_tensor_chunks, timed_resources
from tpeanuts.core.common.oscillation import OscillationParameters
from tpeanuts.config.propagation import PropagationConfig
from tpeanuts.medium.earth.profile import EarthParameters, EarthProfile
from tpeanuts.medium.earth.probability import earth_probability_state
from tpeanuts.medium.solar.profile import SolarProfile
from tpeanuts.medium.solar.probability import solar_probability_mass
from tpeanuts.medium.atmosphere.profile import AtmosphereParameters
from tpeanuts.medium.atmosphere.evolutor import atmosphere_evolutor
from tpeanuts.medium.atmosphere.probability import atmosphere_probability_transition
from tpeanuts.util.context import RuntimeContext

## 2. Paths and Configuration

### 2.1 Paths

`load_notebook_config()` resolves the project and applies the shared plotting/runtime style. All figures and CSV tables are saved under the notebook-relative output path `benchmark/`.

**Expected results:** package and output directories exist and the selected runtime device is reported.

In [ ]:
config = load_notebook_config()
DEVICE, DTYPE = config.device, config.dtype
context = RuntimeContext.resolve(DEVICE, DTYPE)
OUTPUT_DIR = config.output_dir('benchmark')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Package dir : {config.package_dir}')
print(f'Output dir  : {OUTPUT_DIR}')
print(f'Device      : {context.device}   dtype: {context.dtype}')
print(f'CPU threads : {torch.get_num_threads()} intra-op / {torch.get_num_interop_threads()} inter-op')

### 2.2 Configuration

The benchmark scans all Cartesian combinations of the energy- and angular-grid sizes shown below. This intentionally produces expensive large-grid workloads; the notebook is supplied unexecuted so the user can select the desired device and repetition count before running it.

| Parameter | Value | Description |
|-----------|-------|-------------|
| Oscillation preset | `_SM_NUFIT52_NO` | Three-flavour normal ordering |
| Energy-grid sizes | 4, 8, 16, 32 | Colab-safe scaling axis $N_E$ |
| Angular-grid sizes | 4, 8, 16, 32 | Colab-safe scaling axis $N_\theta$ |
| Warm-up executions | 1 | Per method and scenario |
| Timed repetitions | 3 | Median reported |
| Earth numerical steps | 400 | Constant-density intervals |
| Atmosphere numerical steps | 600 | Constant-density intervals |
| Atmosphere perturbative fit | 6 cubic segments | Automatically fitted |
| Detector depth | 1000 m | Shared geometry |
| CPU threads | `min(4, os.cpu_count())` | Adapts to the assigned Colab runtime |
| Process RAM budget | `min(10 GiB, 75% total RAM)` | Adapts to the assigned Colab runtime |
| PyTorch VRAM fraction | 70% | Per-process CUDA allocator limit |
| Energy/angle chunks | 16 / 8 | Common to both propagation methods |
| Resource sampling | 20 ms | Process RSS and CPU utilization |

**Expected results:** both methods build finite probability tensors with identical shapes; timing varies with hardware but the speedup definition remains unchanged.

In [ ]:
N_EGRIDS = 4
N_AGRIDS = 4
ENERGY_GRID_SIZES = [4*2**n for n in range(N_EGRIDS)]
ANGLE_GRID_SIZES = [4*2**n for n in range(N_AGRIDS)]
WARMUP, REPEATS = 1, 3
TOTAL_RAM_GB = psutil.virtual_memory().total / 1024**3
MAX_RAM_GB, MAX_VRAM_FRACTION = min(10.0, 0.75 * TOTAL_RAM_GB), 0.70
ENERGY_CHUNK_SIZE, ANGLE_CHUNK_SIZE = 16, 8
RESOURCE_SAMPLE_INTERVAL_S = 0.02
EMPTY_CUDA_CACHE_BETWEEN_CASES = True
DEPTH_M, N_EARTH_STEPS = 1000.0, 400
CDTYPE = torch.complex128 if DTYPE == torch.float64 else torch.complex64
oscillation = PropagationConfig.oscillation_parameters_from_preset('_SM_NUFIT52_NO', context=context, antinu=False)
earth = EarthProfile(params=EarthParameters(profile_perturbative_kwargs={'density_file': str(config.earth_density_file), 'tabulated_density': False}), context=context)
solar = SolarProfile.default(context=context)
atmosphere = AtmosphereParameters(atmosphere_density_source='exponential', nsteps=600, matter=True, perturbative_segments=6, perturbative_degree=3)
print(DEVICE,CDTYPE, DTYPE)
BASIS = torch.eye(3, device=DEVICE, dtype=CDTYPE)
benchmark_tables = []
if context.device.type == 'cuda':
    torch.cuda.set_per_process_memory_fraction(MAX_VRAM_FRACTION, device=context.device)
print(f'RAM total   : {TOTAL_RAM_GB:.1f} GiB')
print(f'RAM budget  : {MAX_RAM_GB:.1f} GiB process RSS')
print(f'Chunks      : energy={ENERGY_CHUNK_SIZE}, angle={ANGLE_CHUNK_SIZE}')
if context.device.type == 'cuda':
    print(f'GPU         : {torch.cuda.get_device_name(context.device)}')
    print(f'VRAM total  : {torch.cuda.get_device_properties(context.device).total_memory/1024**3:.1f} GiB')
    print(f'VRAM limit  : {MAX_VRAM_FRACTION:.0%}')

### 2.3 Benchmark helpers

The helper synchronizes CUDA, warms both paths, processes both methods with identical energy-angle chunks, and reduces each result to a finite checksum to force materialization without retaining the complete grid. A background sampler records process RAM and CPU consumption, while CUDA peak statistics provide allocated and reserved VRAM. Each section reports timing, speedup and resource peaks.

**Expected results:** each scenario produces 16 rows (four energy sizes times four angular sizes) with positive timings, finite speedups and resource measurements below the configured budgets. Smaller chunks reduce peak memory at the cost of additional dispatch overhead.

In [ ]:
def grid_chunks(energy, angles):
    return iter_grid_chunks(energy, angles, energy_chunk_size=ENERGY_CHUNK_SIZE, angle_chunk_size=ANGLE_CHUNK_SIZE, max_ram_gb=MAX_RAM_GB)

def reduce_chunks(chunks):
    return reduce_tensor_chunks(chunks, device=DEVICE, dtype=DTYPE, max_ram_gb=MAX_RAM_GB)

def timed(fn, method):
    return timed_resources(lambda: fn(method), device=DEVICE, repeats=REPEATS, warmups=WARMUP, max_ram_gb=MAX_RAM_GB, sample_interval_s=RESOURCE_SAMPLE_INTERVAL_S, empty_cuda_cache=EMPTY_CUDA_CACHE_BETWEEN_CASES, cpu_normalization_cores=CPU_THREADS)

def benchmark_grid(name, workload_factory, filename):
    records = []
    for n_energy in ENERGY_GRID_SIZES:
        print('Energy Grid Size:',n_energy)
        for n_angle in ANGLE_GRID_SIZES:
            print('.',end='')
            fn = workload_factory(n_energy, n_angle)
            samples_a, resources_a = timed(fn, 'analytical')
            samples_n, resources_n = timed(fn, 'numerical')
            t_a, t_n = np.median(samples_a), np.median(samples_n)
            records.append({'case': name, 'n_energy': n_energy, 'n_angle': n_angle, 'perturbative_s': t_a, 'numerical_s': t_n, 'speedup': t_n/t_a, 'perturbative_std_s': samples_a.std(), 'numerical_std_s': samples_n.std(), **{f'perturbative_{key}': value for key, value in resources_a.items()}, **{f'numerical_{key}': value for key, value in resources_n.items()}})
        print()
    frame = pd.DataFrame(records); benchmark_tables.append(frame); display(frame)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
    for ax, column, title in zip(axes, ('perturbative_s', 'numerical_s', 'speedup'), ('Perturbative time [s]', 'Numerical time [s]', 'Speedup numerical / perturbative')):
        pivot = frame.pivot(index='n_angle', columns='n_energy', values=column)
        values = pivot.to_numpy(); norm = mcolors.LogNorm(vmin=values.min(), vmax=values.max()) if values.max() > values.min() else None
        image = ax.imshow(values, origin='lower', aspect='auto', cmap='viridis' if column != 'speedup' else 'RdYlGn', norm=norm)
        ax.set_xticks(range(len(pivot.columns)), pivot.columns); ax.set_yticks(range(len(pivot.index)), pivot.index)
        ax.set(xlabel='Energy-grid size', ylabel='Angular-grid size', title=title); fig.colorbar(image, ax=ax)
        if column == 'speedup':
            for row in range(values.shape[0]):
                for col in range(values.shape[1]): ax.text(col, row, f'{values[row,col]:.1f}x', ha='center', va='center', fontsize=7)
    fig.suptitle(name); fig.tight_layout(); save_and_show(filename, fig, output_dir=OUTPUT_DIR, show_plots=config.show_plots)
    resource_filename = filename.replace('_scaling.png', '_resource_scaling.png')
    plot_grid_resource_scaling(frame, title=name, filename=resource_filename, output_dir=OUTPUT_DIR, show_plots=config.show_plots)
    return frame

## 3. Pure Flavour States through Earth

The workload propagates all three pure flavour states over a common energy–nadir grid. The analytical path evaluates every energy-angle chunk as one broadcast tensor, whereas the numerical path retains its required scalar-trajectory angular loop.

**Expected results:** perturbative propagation should avoid hundreds of local matrix exponentials per trajectory and therefore achieve a speedup, particularly on CPU. The accompanying 2×4 resource grid shows peak RAM, allocated VRAM, normalized CPU and GPU utilization versus energy-grid size (top) and angular-grid size (bottom).

In [ ]:
def earth_workload_factory(n_energy, n_angle):
    energy = torch.logspace(np.log10(100.0), np.log10(2.0e4), n_energy, device=DEVICE, dtype=DTYPE)
    angles = torch.linspace(0.05, 1.50, n_angle, device=DEVICE, dtype=DTYPE)
    def workload(method):
        def chunks():
            for energy_chunk, angle_chunk in grid_chunks(energy, angles):
                for state in BASIS:
                    if method == 'analytical':
                        yield earth_probability_state(state, earth, oscillation, energy_chunk[:, None], angle_chunk[None, :], DEPTH_M, method=method, massbasis=False, reunitarize=True)
                    else:
                        for eta in angle_chunk:
                            yield earth_probability_state(state, earth, oscillation, energy_chunk[:, None], eta, DEPTH_M, method=method, massbasis=False, nsteps=N_EARTH_STEPS, context=context)
        return reduce_chunks(chunks())
    return workload

earth_df = benchmark_grid('Pure flavour through Earth', earth_workload_factory, 'benchmark1_fig1_earth_scaling.png')

## 4. Solar Neutrinos to the Detector

Solar $^8$B mass weights are precomputed outside the timed region; the benchmark measures only their Earth propagation to the detector. The analytical path broadcasts each complete energy–nadir chunk, while the numerical path loops over scalar trajectories.

**Expected results:** the relative advantage follows the Earth workload, while the absolute time is lower because only one incoherent state is propagated instead of three pure states. The accompanying 2×4 resource grid shows peak RAM, allocated VRAM, normalized CPU and GPU utilization versus energy-grid size (top) and angular-grid size (bottom).

In [ ]:
def solar_workload_factory(n_energy, n_angle):
    energy = torch.linspace(0.5, 15.0, n_energy, device=DEVICE, dtype=DTYPE)
    angles = torch.linspace(0.10, 1.45, n_angle, device=DEVICE, dtype=DTYPE)
    weights = solar_probability_mass(oscillation, energy, solar, '8B')
    def workload(method):
        def chunks():
            for energy_chunk, angle_chunk in grid_chunks(energy, angles):
                e0 = int(torch.searchsorted(energy, energy_chunk[0]).item())
                weight_chunk = weights[e0:e0 + energy_chunk.numel()]
                if method == 'analytical':
                    yield earth_probability_state(weight_chunk, earth, oscillation, energy_chunk[:, None], angle_chunk[None, :], DEPTH_M, method=method, massbasis=True, reunitarize=True)
                else:
                    for eta in angle_chunk:
                        yield earth_probability_state(weight_chunk, earth, oscillation, energy_chunk[:, None], eta, DEPTH_M, method=method, massbasis=True, nsteps=N_EARTH_STEPS, context=context)
        return reduce_chunks(chunks())
    return workload

solar_df = benchmark_grid('Solar production to detector', solar_workload_factory, 'benchmark1_fig2_solar_detector_scaling.png')

## 5. Atmospheric Neutrinos to the Surface

The complete atmosphere transition matrix is computed in one batched call over energy and zenith angle. Profile construction and automatic polynomial fitting are included in the perturbative timing.

**Expected results:** six fitted perturbative segments should require substantially fewer matrix operations than 600 numerical segments, producing the largest speedup of the notebook. The accompanying 2×4 resource grid shows peak RAM, allocated VRAM, normalized CPU and GPU utilization versus energy-grid size (top) and angular-grid size (bottom).

In [ ]:
def atmosphere_surface_workload_factory(n_energy, n_angle):
    energy = torch.logspace(2.0, 5.0, n_energy, device=DEVICE, dtype=DTYPE)
    angles = torch.linspace(0.0, 180.0, n_angle, device=DEVICE, dtype=DTYPE)
    def workload(method):
        return reduce_chunks(atmosphere_probability_transition(oscillation, e[:, None], 25.0, a[None, :], DEPTH_M/1000.0, atmosphere=atmosphere, context=context, method=method) for e, a in grid_chunks(energy, angles))
    return workload

atmosphere_surface_df = benchmark_grid('Atmosphere production to surface', atmosphere_surface_workload_factory, 'benchmark1_fig3_atmosphere_surface_scaling.png')

## 6. Atmospheric Neutrinos to the Detector

The end-to-end workload computes the batched atmosphere evolutor, preserves coherent amplitudes at the surface, and propagates each initial flavour through Earth. The analytical Earth stage keeps the joint energy-angle batch; only the numerical Earth stage loops over scalar trajectories.

**Expected results:** total speedup combines the atmosphere advantage with the more moderate Earth advantage. Up-going trajectories dominate numerical execution time. The accompanying 2×4 resource grid shows peak RAM, allocated VRAM, normalized CPU and GPU utilization versus energy-grid size (top) and angular-grid size (bottom).

In [ ]:
def atmosphere_detector_workload_factory(n_energy, n_angle):
    energy = torch.logspace(2.0, 4.5, n_energy, device=DEVICE, dtype=DTYPE)
    zenith = torch.linspace(30.0, 180.0, n_angle, device=DEVICE, dtype=DTYPE)
    def workload(method):
        def chunks():
            for energy_chunk, zenith_chunk in grid_chunks(energy, zenith):
                S_atm, _ = atmosphere_evolutor(oscillation, energy_chunk[:, None], 25.0, zenith_chunk[None, :], DEPTH_M/1000.0, atmosphere=atmosphere, context=context, method=method)
                eta_chunk = torch.pi - torch.deg2rad(zenith_chunk)
                for state in BASIS:
                    if method == 'analytical':
                        surface_state = torch.matmul(S_atm, state)
                        yield earth_probability_state(surface_state, earth, oscillation, energy_chunk[:, None], eta_chunk[None, :], DEPTH_M, method=method, massbasis=False, reunitarize=True)
                    else:
                        for i_angle, eta in enumerate(eta_chunk):
                            surface_state = torch.matmul(S_atm[:, i_angle], state)
                            yield earth_probability_state(surface_state, earth, oscillation, energy_chunk[:, None], eta, DEPTH_M, method=method, massbasis=False, nsteps=N_EARTH_STEPS, context=context)
                del S_atm
        return reduce_chunks(chunks())
    return workload

atmosphere_detector_df = benchmark_grid('Atmosphere production to detector', atmosphere_detector_workload_factory, 'benchmark1_fig4_atmosphere_detector_scaling.png')

## 7. Resource Consumption

The plots compare the maximum process RAM, allocated/reserved VRAM and process CPU utilization observed in each physical section. Peaks are taken over every grid size and timed repetition, so they represent the most demanding measured configuration rather than an average. CPU utilization is normalized by `CPU_THREADS`, so 100% represents full use of the configured CPU budget.

**Expected results:** process RAM remains below `MAX_RAM_GB`; CUDA memory remains below the configured allocator fraction. The numerical method should generally require more memory for its discretized trajectories, while CPU utilization remains bounded by the configured thread pools.

In [ ]:
resource_results = pd.concat(benchmark_tables, ignore_index=True)
resource_columns = ['ram_gb', 'vram_allocated_gb', 'vram_reserved_gb', 'cpu_percent', 'gpu_percent']
resource_summary = pd.DataFrame([{'case': case, 'method': method, **{metric: group[f'{method}_{metric}'].max() for metric in resource_columns}} for case, group in resource_results.groupby('case') for method in ('perturbative', 'numerical')])
#display(resource_summary)
resource_summary.to_csv(OUTPUT_DIR / 'benchmark1_resource_summary.csv', index=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
x = np.arange(resource_summary['case'].nunique()); width = 0.38
cases = resource_summary['case'].drop_duplicates().tolist()
for ax, metric, title in zip(axes.flat, resource_columns, ('Peak process RAM [GiB]', 'Peak allocated VRAM [GiB]', 'Peak reserved VRAM [GiB]', 'Peak process CPU [%]', 'Peak GPU utilization [%]')):
    for offset, method in zip((-width/2, width/2), ('perturbative', 'numerical')):
        values = resource_summary[resource_summary.method == method].set_index('case').loc[cases, metric]
        ax.bar(x + offset, values, width, label=method.capitalize())
    ax.set_xticks(x, [f'Section {index + 3}' for index in range(len(cases))]); ax.set_title(title); ax.grid(axis='y', alpha=.25)
    if metric == 'ram_gb': ax.axhline(MAX_RAM_GB, color='tab:red', ls='--', label='RAM budget')
    ax.legend(fontsize=8)
axes.flat[-1].axis('off')
fig.suptitle('Peak resource consumption by benchmark section'); fig.tight_layout(); save_and_show('benchmark1_fig5_resource_consumption.png', fig, output_dir=OUTPUT_DIR, show_plots=config.show_plots)
display(resource_summary.set_index(['case', 'method']))

## 8. Summary

The combined table contains one comparison for every $(N_E,N_\theta)$ pair and scenario. The aggregate chart shows diagonal scaling, $N_E=N_\theta$, for execution times and speedup. Raw results are exported to `benchmark1_perturbative_vs_numerical.csv`.

**Expected results:** all times and speedups are finite and positive. Speedup depends on device, batch size and trajectory discretization; the atmosphere-only workload is expected to benefit most from replacing 600 numerical steps with six perturbative segments.

In [ ]:
results = pd.concat(benchmark_tables, ignore_index=True)
display(results)
results.to_csv(OUTPUT_DIR / 'benchmark1_perturbative_vs_numerical.csv', index=False)
numeric_results = results.select_dtypes(include=np.number)
assert np.isfinite(numeric_results.to_numpy()).all() and (results[['perturbative_s', 'numerical_s', 'speedup']] > 0).all().all()
assert (results[['perturbative_ram_gb', 'numerical_ram_gb']] <= MAX_RAM_GB).all().all()

diagonal = results[results.n_energy == results.n_angle]
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0))
for name, group in diagonal.groupby('case'):
    axes[0].plot(group.n_energy, group.perturbative_s, 'o-', label=f'{name} — perturbative')
    axes[0].plot(group.n_energy, group.numerical_s, 'o--', label=f'{name} — numerical')
    axes[1].plot(group.n_energy, group.speedup, 'o-', label=name)
for ax in axes: ax.set_xscale('log', base=2); ax.set_xticks(ENERGY_GRID_SIZES, ENERGY_GRID_SIZES); ax.grid(alpha=.25)
axes[0].set_yscale('log'); axes[0].set(xlabel='N_E = N_angle', ylabel='Median execution time [s]', title='Diagonal timing scaling'); axes[0].legend(fontsize=7)
axes[1].axhline(1.0, color='0.3', ls='--'); axes[1].set_yscale('log'); axes[1].set(xlabel='N_E = N_angle', ylabel='Speedup numerical / perturbative', title='Diagonal speedup scaling'); axes[1].legend(fontsize=8)
fig.suptitle('Perturbative versus numerical grid-size benchmark'); fig.tight_layout(); save_and_show('benchmark1_fig6_summary_scaling.png', fig, output_dir=OUTPUT_DIR, show_plots=config.show_plots)
print('All four benchmark scenarios completed successfully.')

### Physical and computational interpretation

The speedup is a property of both the algorithm and the workload. Numerical propagation scales with the number of sampled intervals and repeatedly evaluates matrix exponentials; perturbative propagation replaces those samples with a small number of fitted segments and analytical residual integrals. Earth speedup is limited by shell geometry and spectral calculations, whereas smooth atmospheric profiles can exploit a much larger reduction in segment count. Absolute timings should therefore always be reported together with grid size, numerical steps, device and dtype.